# Run Graph-Mamba Inference

Inference parameters come from `configs/default.yaml`. The notebook reads the global `run_name`, loads `checkpoints/<run_name>/best_model.pt`, reads graph data from `output/<run_name>/`, and writes a compiled AirNow-like NetCDF file under `output/<run_name>/`.

In [ ]:
from pathlib import Path
import copy
import json
import sys

REPO_ROOT = Path.cwd()
if not (REPO_ROOT / 'configs/default.yaml').exists():
    REPO_ROOT = Path('/mnt/data3/GraphMamba')

CONFIG_PATH = REPO_ROOT / 'configs/default.yaml'
sys.path.insert(0, str(REPO_ROOT / 'src'))

from graphmamba.config import load_config, resolve_inference_paths

def make_repo_path(value):
    path = Path(value)
    return str(path if path.is_absolute() else REPO_ROOT / path)

config = copy.deepcopy(load_config(CONFIG_PATH))
config['paths']['checkpoint_root'] = make_repo_path(config['paths']['checkpoint_root'])
config['paths']['output_root'] = make_repo_path(config['paths']['output_root'])
inference_cfg = config['inference']
inference_paths = resolve_inference_paths(config)
checkpoint_path = inference_paths['checkpoint_path']
graph_path = inference_paths['graph_path']
output_csv = inference_paths['output_csv']
output_netcdf = inference_paths['output_netcdf']

print(json.dumps({
    'config_path': str(CONFIG_PATH),
    'run_name': config['run_name'],
    'checkpoint_path': str(checkpoint_path),
    'graph_path': str(graph_path),
    'date_range': [inference_cfg['start_date'], inference_cfg['end_date']],
    'device': inference_cfg['device'],
    'multi_gpu': inference_cfg.get('multi_gpu', 'off'),
    'gpu_min_free_memory_gb': inference_cfg.get('gpu_min_free_memory_gb', 0.0),
    'gpu_max_count': inference_cfg.get('gpu_max_count'),
    'batch_size': inference_cfg['batch_size'],
    'show_progress': inference_cfg.get('show_progress', True),
    'output_csv': str(output_csv),
    'output_netcdf': str(output_netcdf),
}, indent=2))

In [ ]:
from graphmamba.inference import load_inference_bundle

bundle = load_inference_bundle(
    checkpoint_path=checkpoint_path,
    graph_path=graph_path,
    device=inference_cfg['device'],
    multi_gpu=inference_cfg.get('multi_gpu', 'auto'),
    gpu_min_free_memory_gb=inference_cfg.get('gpu_min_free_memory_gb', 0.0),
    gpu_max_count=inference_cfg.get('gpu_max_count'),
)
train_config = bundle.config
print(json.dumps({
    'device': str(bundle.device),
    'device_ids': bundle.device_ids,
    'target_mean': bundle.target_mean,
    'target_std': bundle.target_std,
    'num_airnow': bundle.graph['metadata']['num_airnow'],
    'num_tempo': bundle.graph['metadata']['num_tempo'],
}, indent=2))

In [ ]:
from graphmamba.data import TempoAirNowDataset

dataset = TempoAirNowDataset(
    graph_path=graph_path,
    tempo_dir=train_config['paths']['tempo_dir'],
    airnow_dir=train_config['paths']['airnow_dir'],
    start_date=inference_cfg['start_date'],
    end_date=inference_cfg['end_date'],
    context_steps=train_config['data']['context_steps'],
    max_time_snap_hours=train_config['data']['max_time_snap_hours'],
    max_column=train_config['data']['max_column'],
    min_valid_targets=inference_cfg['min_valid_targets'],
    target_mean=bundle.target_mean,
    target_std=bundle.target_std,
)
print(json.dumps({
    'inference_windows': len(dataset),
    'start_date': inference_cfg['start_date'],
    'end_date': inference_cfg['end_date'],
}, indent=2))

In [ ]:
from graphmamba.inference import predict_dataframe

predictions = predict_dataframe(
    dataset,
    bundle,
    batch_size=inference_cfg['batch_size'],
    num_workers=inference_cfg['num_workers'],
    show_progress=inference_cfg.get('show_progress', True),
)
predictions.head()

In [ ]:
from graphmamba.metrics import pearson_corr

observed = predictions[predictions['has_observation']].copy()
if len(observed):
    rmse = ((observed['predicted_no2_ppb'] - observed['observed_no2_ppb']) ** 2).mean() ** 0.5
    ss_res = ((observed['observed_no2_ppb'] - observed['predicted_no2_ppb']) ** 2).sum()
    ss_tot = ((observed['observed_no2_ppb'] - observed['observed_no2_ppb'].mean()) ** 2).sum()
    r2 = 1.0 - ss_res / ss_tot if ss_tot > 0 else float('nan')
    dt_corr = pearson_corr(observed['delta_t_hours'].to_numpy(), observed['absolute_error_ppb'].to_numpy())
    distance_corr = pearson_corr(observed['nearest_tempo_distance_km'].to_numpy(), observed['absolute_error_ppb'].to_numpy())
    print(json.dumps({
        'observations': int(len(observed)),
        'rmse': float(rmse),
        'r2': float(r2),
        'abs_error_delta_t_corr': float(dt_corr),
        'abs_error_nearest_tempo_distance_corr': float(distance_corr),
    }, indent=2))
else:
    print('No observed AirNow targets are available in this inference range.')

In [ ]:
from graphmamba.inference import write_predictions_netcdf

netcdf_path = write_predictions_netcdf(
    predictions,
    output_netcdf,
    run_name=config['run_name'],
    attrs={
        'checkpoint_path': checkpoint_path,
        'graph_path': graph_path,
        'inference_start_date': inference_cfg['start_date'],
        'inference_end_date': inference_cfg['end_date'],
    },
)
output_csv.parent.mkdir(parents=True, exist_ok=True)
predictions.to_csv(output_csv, index=False)
print({'netcdf': str(netcdf_path), 'csv': str(output_csv)})